In [124]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
from datetime import timedelta
import logging
import mysql

In [125]:
# Prepare the DataFrame for the investment window analysis by ensuring proper data types and sorting.
def prepare_data(df, asset_class=None, start=pd.Timestamp.now(), end=pd.DateOffset(months=3)):
    """
    Prepare and clean the DataFrame for analysis.
    Args:
        df: DataFrame with 'TransactionDate', 'TransactionClass', and 'Available' columns
        asset_class: String to filter TransactionClass column (e.g., 'US Agencies')
        start_date_parameter: Starting date for analysis, defaults to today
        end_date_parameter: Ending date for analysis, defaults to 3 months from today
    Returns:
        DataFrame: Cleaned and sorted DataFrame
    """
    # Define start and end dates
    start_date = pd.Timestamp(start).normalize()
    end_date = pd.Timestamp(end).normalize()

    # 1. Filter the data within the date range
    data = df[(df['TransactionDate'] >= start_date) & (df['TransactionDate'] <= end_date)].copy()

    # 2. Filter by asset class if specified
    if asset_class is not None:
        # Check if asset_class exists in TransactionClass column
        if asset_class not in df['TransactionClass'].unique():
            raise ValueError(f"Asset class '{asset_class}' not found in data")
        data = data[data['TransactionClass'] == asset_class].copy()
        if data.empty:
            raise ValueError(f"No data found for asset class: {asset_class}")

    # Set the TransactionDate to datetime
    data['TransactionDate'] = pd.to_datetime(data['TransactionDate'])

    # Sort
    data = data.sort_values('TransactionDate').reset_index(drop=True)
    data['TransactionClass'] = asset_class if asset_class else 'Not Specified'
    return data[['TransactionDate', 'Available','TransactionClass']]

In [126]:
"""
Determine investment windows where the available balance exceeds a threshold and track the duration of these windows.
df: DataFrame with 'TransactionDate' and 'Available' columns
threshold = 1,000,000 - minimum balance to consider
Returns the investment windows as a DataFrame with columns:
    'start_date': Start date of the investment window
    'end_date': End date of the investment window (None if ongoing)
    'amount': The balance amount during the window
    'duration': Duration of the window in days
"""
# Function to detect window intervals
# Define the function
def find_window_intervals(df, threshold=1_000_000):
    df = df.sort_values('TransactionDate').reset_index(drop=True)

    # Step 1: Get sorted unique amounts >= threshold
    unique_amounts = sorted(df[df['Available'] >= threshold]['Available'].unique())

    intervals = []

    for amount in unique_amounts:
        in_interval = False
        start_date = None
        duration = 0

        for i in range(len(df)):
            date = df.loc[i, 'TransactionDate']
            available = df.loc[i, 'Available']

            if available >= amount:
                if not in_interval:
                    start_date = date
                    in_interval = True
                    duration = 1
                else:
                    duration += 1
            else:
                if in_interval:
                    end_date = date
                    intervals.append({
                        'start_date': start_date,
                        'end_date': end_date,
                        'amount': amount,
                        'duration': duration
                    })
                    in_interval = False
                    start_date = None
                    duration = 0

        # If still in an interval at the end
        if in_interval:
            end_date = df['TransactionDate'].iloc[-1]
            intervals.append({
                'start_date': start_date,
                'end_date': end_date,
                'amount': amount,
                'duration': duration
            })

    # Create DataFrame
    df_intervals = pd.DataFrame(intervals)

    # Format columns
    df_intervals['start_date'] = pd.to_datetime(df_intervals['start_date']).dt.date
    df_intervals['end_date'] = pd.to_datetime(df_intervals['end_date']).dt.date
    df_intervals['amount'] = df_intervals['amount'].round(2)
    df_intervals['change'] = (df_intervals['amount'] - df_intervals['amount'].shift(1).fillna(0)).round(2)

    return df_intervals

In [127]:
# Step 2: Keep Only the Max Amount Interval for Each Overlapping Period
def drop_overlapping_lower_intervals(df):
    # Group by start_date and end_date, then get the row with maximum available amount
    result = df.loc[df.groupby(['start_date', 'end_date'])['amount'].idxmax()]
    return result

In [128]:
# Step 3: Update Changed Amounts Based on Prior Date from the Original Data
def update_changed_amounts(df_intervals, df_data):

    # Compute the day before each interval start
    df_intervals['lookup_date'] = df_intervals['start_date'] - pd.Timedelta(days=1)
    # Merge with df_data on this lookup date
    df_result = df_intervals.merge(
        df_data.rename(columns={'Date': 'lookup_date', 'Amount': 'prev_amount'}), on='lookup_date', how='left'
    )
    return df_result


In [129]:
# MAIN
#     analyze_balance_data() for running the asset class balances processing

# In prod we'll be looping over each asset class and process the investment windows one by one
asset_classes = ['Certificate of Deposit', 'Mutual Fund', 'Commercial Paper', 'Money Market', 'US Treasuries', 'US Agencies']
asset_index = 5
# Load and print the data
running_balances = pd.read_pickle('running_balances.pkl')
data = prepare_data(running_balances, asset_classes[asset_index], '2025-09-04', '2025-12-31')

# with pd.option_context('display.max_columns', None,
#                        'display.max_rows', None,
#                        'display.width', None,
#                        'display.expand_frame_repr', False):
#     pd.options.display.float_format = '${:,.2f}'.format
# display(data.head(30))

# data.info()

all_intervals = find_window_intervals(data).sort_values(['start_date'], ascending=[True])
intervals = drop_overlapping_lower_intervals(all_intervals)

# final_intervals = update_changed_amounts(intervals, data)

# Debug display
print("\nInvestment windows:", len(intervals))
with pd.option_context('display.max_columns', None,
                       'display.max_rows', 100,
                       'display.width', None,
                       'display.expand_frame_repr', False):
    pd.options.display.float_format = '${:,.2f}'.format
display(intervals.head(25))


Investment windows: 47


,start_date,end_date,amount,duration,change
6,2025-09-04,2025-09-24,"$4,642,791.81",20,"$2,320,667.26"
16,2025-09-05,2025-09-10,"$10,282,791.81",5,"$3,355,600.00"
11,2025-09-05,2025-09-24,"$6,927,191.81",19,"$2,284,400.00"
82,2025-09-08,2025-09-10,"$23,637,591.81",2,"$593,966.30"
100,2025-09-09,2025-09-10,"$28,637,591.81",1,"$1,317,467.26"
188,2025-09-11,2025-09-12,"$64,013,191.81",1,"$415,600.00"
144,2025-09-11,2025-09-15,"$47,403,991.81",4,"$1,000,000.00"
140,2025-09-11,2025-09-24,"$46,403,991.81",13,"$11,824,400.00"
201,2025-09-16,2025-09-17,"$65,710,791.81",1,"$1,069,166.30"
193,2025-09-16,2025-09-19,"$64,457,591.81",3,"$444,400.00"
